# Pandas — Structures and Loading

Notebook 02 left you with fast typed arrays — and a list of what NumPy was missing: **labels**, **mixed types in one table**, and a **first-class missing-data story**. Pandas adds all three on top of NumPy.

This notebook covers four things.

1. **`Series`** — a labeled 1D array (one column of a table)
2. **`DataFrame`** — a labeled 2D table built from `Series` columns
3. **`.loc` vs `.iloc`** — label-based vs position-based selection (the single most confusing pandas concept)
4. **Reading data** — CSV, JSON, parquet, and SQL — getting real data into a DataFrame

We finish notebook 03 ready to clean and transform data in notebook 04.


## `Series` — a labeled 1D array

A `Series` is a NumPy array bolted to an **index**. The values are still a typed NumPy block underneath — `Series.values` returns the raw `ndarray`. The new thing is that every value has a **label**: a customer ID, a date, a city name, anything you choose.

For a JVM reader: a `Series` is closer to a `LinkedHashMap<Key, Value>` than to a `List<Value>`. The order is preserved, lookup by key is cheap, and iteration is in insertion order. The big difference from a Java map is that the *values* are stored in a contiguous typed block (the NumPy array underneath), so arithmetic across the whole Series still runs at NumPy speed.

If you don't supply an index, pandas gives you the integer positions `0, 1, 2, ...` — which makes a `Series` look just like a NumPy array. The index is always there; it just may be the trivial one.


In [ ]:
import numpy as np
import pandas as pd

monthly_income = pd.Series(
    [45000, 82000, 120000, 67000, 155000],
    index=[1001, 1002, 1003, 1004, 1005],
    name="monthly_income",
)
print(monthly_income)
print()
print("values (the underlying ndarray):", monthly_income.values)
print("index:", monthly_income.index.tolist())
print("dtype:", monthly_income.dtype)
print("name: ", monthly_income.name)


## Selecting from a Series — `.loc` vs `.iloc`

Two ways to grab a value out of a `Series`.

- **`.loc[label]`** — look up by the index label. Customer 1003's income.
- **`.iloc[position]`** — look up by integer position. The third row, regardless of label.

If your index *is* integers, the two look similar but are not the same. `s.loc[1003]` asks for the row labeled `1003`. `s.iloc[1003]` asks for the row at position `1003` — which would crash if the Series only has 5 rows.

JVM analogy: `.loc` is `map.get(key)`. `.iloc` is `list.get(position)`. Use the one that matches what you mean.

**Slice gotcha:** `.loc` slices are **inclusive** on the right; `.iloc` slices are **exclusive** on the right, like Python lists.


In [ ]:
# By label
print("loc[1003]:", monthly_income.loc[1003])

# By position
print("iloc[2]: ", monthly_income.iloc[2])

# Slices — loc is INCLUSIVE on the right, iloc is EXCLUSIVE
print("\nloc[1002:1004]  (inclusive of 1004):")
print(monthly_income.loc[1002:1004])

print("\niloc[1:4]  (exclusive of position 4):")
print(monthly_income.iloc[1:4])


## `DataFrame` — a labeled 2D table

A `DataFrame` is a collection of `Series` sharing the **same index** — each column is one Series, and they all line up row-by-row. Each column has its own dtype, which is how pandas fits mixed types in one structure. NumPy could not.

Attributes to know on day one.

| Attribute | What it gives you |
|---|---|
| `df.shape` | `(rows, cols)` tuple |
| `df.columns` | The column names |
| `df.index` | The row labels |
| `df.dtypes` | The dtype of each column |
| `df.head()` / `df.tail()` | First / last 5 rows |
| `df.info()` | Per-column type and null counts |
| `df.describe()` | Numeric summary per column |


In [ ]:
customers = pd.DataFrame({
    "name": ["Aarav Sharma", "Diya Patel", "Vihaan Reddy", "Ananya Iyer", "Arjun Nair"],
    "age": [27, 34, 45, 29, 52],
    "city": ["Bengaluru", "Mumbai", "Hyderabad", "Chennai", "Bengaluru"],
    "monthly_income": [45000, 82000, 120000, 67000, 155000],
}, index=[1001, 1002, 1003, 1004, 1005])
customers.index.name = "customer_id"

print(customers)
print("\nshape:  ", customers.shape)
print("columns:", customers.columns.tolist())
print("dtypes:")
print(customers.dtypes)


## Selecting columns

Two patterns cover all the column selection you will do.

- `df["col"]` — returns a single column as a `Series`.
- `df[["col1", "col2"]]` — returns a `DataFrame` with just those columns (note the double brackets).

The double-bracket trick trips up everyone once. The outer brackets are the indexer; the inner brackets build a Python list of column names. Forget the inner brackets and pandas thinks you handed it a tuple, which it does not know how to interpret.


In [ ]:
# Single column — returns a Series
print(customers["name"])
print("type:", type(customers["name"]).__name__)

# Multiple columns — returns a DataFrame
print()
print(customers[["name", "city"]])
print("type:", type(customers[["name", "city"]]).__name__)


## Selecting rows — `.loc`, `.iloc`, and boolean masks

Three ways. Each has its place.

1. **`df.loc[label]`** — pick a row by its index label.
2. **`df.iloc[position]`** — pick a row by its integer position.
3. **`df[boolean_mask]`** — keep rows where a condition is true.

You can also combine row and column selection in one shot with `df.loc[rows, cols]` or `df.iloc[positions, col_positions]`. That is usually clearer than chained `[]` access — and it avoids the dreaded `SettingWithCopyWarning` later when you start assigning.


In [ ]:
# Single row by label — returns a Series (the row, transposed)
print("loc[1003]:")
print(customers.loc[1003])

# Single row by position
print("\niloc[0]:")
print(customers.iloc[0])

# Row + column slice in one shot
print("\nloc[1002:1004, ['name', 'city']]:")
print(customers.loc[1002:1004, ["name", "city"]])


In [ ]:
# Boolean mask — a Series of True/False, one entry per row
mask = customers["age"] > 40
print("mask:")
print(mask)

# Apply it to the DataFrame
print("\ncustomers older than 40:")
print(customers[mask])

# Combine conditions with & (and) / | (or). Each condition needs its own parens.
print("\nolder than 40 AND in Bengaluru:")
print(customers[(customers["age"] > 40) & (customers["city"] == "Bengaluru")])


## Reading data — CSV, JSON, parquet, SQL

The four formats you will meet most often, and the pandas function for each.

| Format | Read | Write | When to use |
|---|---|---|---|
| **CSV** | `pd.read_csv` | `df.to_csv` | Text, human-readable, universal. Type info is lost — every column is a string until pandas guesses. |
| **JSON** | `pd.read_json` | `df.to_json` | Nested data, REST API responses, configuration. Slower than CSV at scale. |
| **Parquet** | `pd.read_parquet` | `df.to_parquet` | Columnar binary format. Preserves dtypes, compresses well, fast to read. The data engineer's default — `apache-spark` and `databricks` write parquet by default. |
| **SQL** | `pd.read_sql` | `df.to_sql` | Anything in a database. Takes a connection object (sqlite, postgres, mysql) and a query string or a table name. |

We will round-trip the `customers` DataFrame through each format on a temp directory so you can see the dtypes survive (or not).


In [ ]:
import tempfile, os

tmp = tempfile.mkdtemp()
csv_path = os.path.join(tmp, "customers.csv")

# Write
customers.to_csv(csv_path)

# Read — by default the index is written as a column, so tell pandas which column to use as the index
back_csv = pd.read_csv(csv_path, index_col="customer_id")
print(back_csv)
print("\ndtypes after CSV round trip:")
print(back_csv.dtypes)


In [ ]:
# JSON — orient='table' preserves dtypes and the index
json_path = os.path.join(tmp, "customers.json")
customers.to_json(json_path, orient="table", indent=2)
back_json = pd.read_json(json_path, orient="table")
print("JSON dtypes:")
print(back_json.dtypes)

# Parquet — preserves dtypes natively (requires pyarrow or fastparquet installed)
parquet_path = os.path.join(tmp, "customers.parquet")
customers.to_parquet(parquet_path)
back_parquet = pd.read_parquet(parquet_path)
print("\nParquet dtypes:")
print(back_parquet.dtypes)


## Reading from SQL

`read_sql` takes a **connection** and a **query**. The connection can be anything that follows the Python database API: `sqlite3.connect(...)` from the standard library, `psycopg2.connect(...)` for Postgres, or a SQLAlchemy engine for anything else.

For a portable demo we use `sqlite3` with an in-memory database — no file written, no driver installed. The pattern is the same against a real warehouse; you just swap the connection.

Two-line shape to remember.

- `df.to_sql(table_name, conn)` — push a DataFrame into a table.
- `pd.read_sql(query_or_table, conn)` — pull a query result back as a DataFrame.


In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")  # in-memory database, no file written

# Write the DataFrame as a SQL table
customers.to_sql("customers", conn, index=True, index_label="customer_id")

# Read it back via a query
result = pd.read_sql(
    "SELECT customer_id, name, city, monthly_income FROM customers WHERE age > 40",
    conn,
    index_col="customer_id",
)
print(result)

conn.close()


## What's next

You can now build a `DataFrame`, pull rows and columns out of it three different ways, and read from the four formats that cover almost every real-world data source.

What you cannot yet do well: **clean and reshape** data once it lands. Real data has missing values, duplicate rows, inconsistent types, and the wrong shape. Notebook 04 covers `dropna` / `fillna`, `groupby`, `merge`, `pivot` / `melt`, and time series — the muscle that gets messy data into a model-ready shape.

Two from-memory exercises before you move on.

1. Build a `loan_accounts` DataFrame with 6 rows and the columns `loan_id`, `customer_id`, `product`, `principal`, `emi`, and `tenure_months`. Index it by `loan_id`. Then write it to a parquet file and read it back — confirm the dtypes survived the round trip.
2. From that DataFrame, use a boolean mask combined with `.loc` to select only the home loans (`product == "home_loan"`) and return just the `principal` and `emi` columns.
